# Asistente conversacional sobre el Código Nacional de Tránsito con RAG

**Maestría en Inteligencia Artificial Aplicada — Universidad Icesi**<br>
**Procesamiento de Lenguaje Natural · Taller de la semana 4**

**Integrantes:** William Alberto Suaza Losada · Anderson Trujillo · Bairon Gutiérrez

## 1. El problema

Casi todos en Colombia hemos tenido alguna duda de tránsito: si el parrillero de la moto también debe usar casco, cuánto cuesta pasarse un semáforo en rojo o cada cuánto le toca la revisión técnico-mecánica a un carro. La respuesta está en la **Ley 769 de 2002, el Código Nacional de Tránsito Terrestre**, pero leerla no es sencillo. Tiene alrededor de 180 artículos, muchos de ellos modificados por leyes y decretos posteriores, y las multas no aparecen en pesos sino en salarios mínimos diarios (SMLDV), repartidas en literales como "C.29" dentro de un único artículo de más de dos mil palabras.

Un modelo de lenguaje por sí solo tampoco resuelve el problema. Puede contestar con mucha seguridad, citar un artículo que no existe o mezclar normas de otros países. Lo que queremos es un asistente que responda en lenguaje sencillo **y** diga en qué artículo se apoya, para que la persona pueda comprobarlo. Por eso planteamos un sistema de **generación aumentada por recuperación (RAG)**: primero se buscan los fragmentos de la ley más relacionados con la pregunta y después un modelo de lenguaje redacta la respuesta usando solo esos fragmentos.

Creemos que RAG encaja bien con este caso por tres razones:

1. **El corpus es cerrado y oficial.** No hace falta que el modelo "sepa" de tránsito; basta con que lea bien el texto que le entregamos.
2. **La ley ya viene organizada en artículos.** Esa estructura permite citar con precisión y verificar si la cita es correcta.
3. **Las normas cambian.** Con RAG, actualizar el asistente es volver a indexar el texto, sin reentrenar ningún modelo.

### Pregunta de investigación

> ¿Qué combinación de fragmentación, modelo de embeddings, número de fragmentos recuperados y re-ranking encuentra mejor los artículos correctos para preguntas coloquiales sobre el Código de Tránsito, y qué modelo generador aprovecha mejor ese contexto para responder con citas correctas y sin inventar?

### Hipótesis

1. **H1.** Fragmentar siguiendo la estructura de la ley (parágrafos, literales y definiciones) recuperará mejor que usar el artículo completo o una ventana fija de palabras, porque respeta las unidades con las que está escrita la norma.
2. **H2.** Los prefijos `query:` y `passage:` de los modelos E5 mejorarán la recuperación, porque así fueron entrenados.
3. **H3.** El re-ranking con un cross-encoder mejorará sobre todo el MRR, es decir, subirá el artículo correcto a las primeras posiciones.
4. **H4.** `gemma3:4b`, que tiene más parámetros, citará mejor que `llama3.2:3b`, a cambio de responder más despacio.
5. **H5.** Un umbral sobre el puntaje de relevancia permitirá rechazar preguntas que no trata el código sin perder muchas preguntas válidas.

---
## 2. Objetivos y alcance

**Objetivo general.** Construir y evaluar un chatbot RAG que responda preguntas ciudadanas sobre la Ley 769 de 2002 citando los artículos en los que se apoya.

**Objetivos específicos**

1. Descargar el texto consolidado de la ley desde el Gestor Normativo de Función Pública y limpiarlo hasta tener un registro por artículo con sus metadatos.
2. Explorar el corpus para decidir cómo fragmentarlo.
3. Construir un set de evaluación propio, con preguntas en lenguaje coloquial y su artículo correcto.
4. Comparar tres estrategias de fragmentación y cuatro variantes de embeddings, con y sin re-ranking y para distintos valores de k.
5. Comparar dos modelos generadores servidos localmente con Ollama.
6. Calibrar un umbral para que el asistente se abstenga cuando la pregunta no está en el código.
7. Integrar todo en un chatbot conversacional con Gradio que muestre las fuentes consultadas.

**Alcance.** Trabajamos solo con la Ley 769 de 2002 consolidada. Quedan por fuera las resoluciones del Ministerio de Transporte, los decretos reglamentarios y las normas locales (pico y placa, impuestos, tarifas). El asistente orienta, pero **no reemplaza una asesoría legal**.

**Limitaciones que conocemos desde el inicio**

- **La compilación oficial no siempre trae la última redacción.** Revisando el texto encontramos que el artículo 22 conserva la vigencia "indefinida" de las licencias de vehículos particulares, aunque la misma página indica que fue modificado por el Decreto 19 de 2012, que la fijó en diez años para menores de 60. El asistente responde según esa compilación y la interfaz lo advierte.
- **El set de evaluación es pequeño** (46 preguntas del código y 6 por fuera) y lo redactamos nosotros. Cada pregunta del código pesa unos 2,2 puntos porcentuales, así que diferencias pequeñas entre configuraciones deben leerse con cuidado.
- **El umbral de abstención se calibra con las mismas seis preguntas por fuera del código** con las que luego lo evaluamos, así que su desempeño real probablemente sea algo menor.

---
## 3. Entorno y reproducibilidad

El cuaderno está pensado para Google Colab con una GPU T4. Primero detectamos si estamos en Colab; solo en ese caso instalamos las dependencias, con versiones fijas para que otra persona obtenga el mismo entorno. Fijamos las semillas de `random`, NumPy y PyTorch, y más adelante los modelos generadores trabajan con temperatura 0 y una semilla fija.

Los datos no se descargan de la página oficial en cada ejecución. Usamos una copia limpia (snapshot) guardada en nuestro repositorio, porque la ley se sigue reformando y la página podría cambiar o no responder. La descarga en vivo queda como respaldo y como forma de comprobar si el texto cambió.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print('Ejecutando en Colab:', IN_COLAB)

In [ ]:
if IN_COLAB:
    !pip install -q "langchain-classic==1.0.8" "langchain-community==0.4.2" "langchain-huggingface==1.2.2" "langchain-ollama==1.1.0" "faiss-cpu==1.15.1" "sentence-transformers==6.1.0" "ollama==0.6.2" "gradio==6.28.0"

In [ ]:
import gc
import hashlib
import io
import json
import os
import random
import re
import shutil
import socket
import ssl
import subprocess
import tempfile
import time
import unicodedata
import warnings
from collections import Counter
from datetime import date
from pathlib import Path
from typing import Any, Optional

import certifi
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from bs4 import BeautifulSoup
from matplotlib.ticker import PercentFormatter

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.dpi'] = 110

print('Dispositivo:', DEVICE, f'({torch.cuda.get_device_name(0)})' if DEVICE == 'cuda' else '')
print('PyTorch:', torch.__version__)

---
## 4. Obtención y limpieza del corpus

Tomamos el texto de la ley del **Gestor Normativo de Función Pública**, que publica la versión consolidada en una sola página e integra las reformas posteriores. Al revisar el HTML vimos que, además del texto vigente, la página trae mucho material que no queremos que el asistente lea como si fuera ley:

- versiones anteriores de cada artículo ocultas tras un botón "Norma Anterior", y bloques "Texto anterior" / "Texto original" que repiten redacciones derogadas (el artículo 131 de multas aparece dos veces por esto);
- notas de vigencia como "Modificado por el art. 21, Ley 1383 de 2010";
- notas de jurisprudencia ("Texto subrayado declarado INEXEQUIBLE por la Corte Constitucional…") y remisiones del tipo "Ver Resolución…".

Las reglas de limpieza que aplicamos son:

1. Eliminar los bloques ocultos y todo lo que viene después de "Texto anterior" o "Texto original" hasta el siguiente artículo.
2. Quitar las notas de vigencia, jurisprudencia y remisiones, pero guardar las leyes y decretos que modificaron cada artículo en el campo `modificado_por`.
3. Cuando una nota dice que el texto **subrayado** fue declarado INEXEQUIBLE, retirar ese texto subrayado, porque ya no tiene efecto.
4. Marcar como no vigentes los artículos declarados inexequibles o derogados en su totalidad, y los que quedan vacíos después de limpiar. No entran al índice, pero los contamos en el análisis exploratorio.
5. Reconocer títulos y capítulos (incluidos los que vienen con letras espaciadas, como "T I T U L O I I") para guardar la ubicación de cada artículo.

Un detalle práctico: el servidor de Función Pública no envía el certificado intermedio de su cadena TLS, y por eso `requests` rechaza la conexión. En lugar de desactivar la verificación con `verify=False`, descargamos el certificado intermedio oficial de la autoridad certificadora y lo agregamos al conjunto de certificados de confianza.

El orden de carga es: archivo local en `../data`, luego el snapshot del repositorio en GitHub y, solo si ambos fallan, la descarga y limpieza en vivo con las mismas funciones.

In [ ]:
FUENTE_URL = 'https://www.funcionpublica.gov.co/eva/gestornormativo/norma.php?i=5557'
# El servidor no envía el certificado intermedio; se añade el oficial de Sectigo sin desactivar la verificación.
INTERMEDIO_URL = 'http://crt.sectigo.com/SectigoRSAOrganizationValidationSecureServerCA.crt'

RE_TITULO = re.compile(r'^T\s*[IÍ]\s*T\s*U\s*L\s*O\s+([IVXL](?:\s*[IVXL])*)\s*\.?$')
RE_CAPITULO = re.compile(r'^CAP[IÍ]TULO\s+([IVXL]+)\s*\.?\s*(.*)$')
RE_ARTICULO = re.compile(r'^A\s?RT[IÍ]CULO\s+(\d+)\s*(?:([A-Z])\b|-\s*(\d+))?\s*[°ºo]?\s*\.?\s*(.*)$')
RE_TRANSITORIO = re.compile(r'^A\s?RT[IÍ]CULO\s+TRANSITOR[IÍ]O\s*\.?\s*(.*)$')
RE_FIN = re.compile(r'^El Presidente del honorable Senado')
RE_OMITIR_DESDE = re.compile(r'^(Texto anterior|Texto original)')
RE_PARRAFO_NOTA = re.compile(
    r'^\(?\s*(Ver\s|Sentencia\s|NOTA:|Declarado\s|Texto subrayado|Norma Anterior|Jurisprudencia)')
RE_NOTA_PARENTESIS = re.compile(
    r'\((?=[^()]*(?:[Mm]odificad|MODIFICAD|[Aa]dicionad|ADICIONAD|[Ss]ustituid|[Dd]erogad|[Rr]eglamentad|\bMod\b|Ver\s))[^()]*\)')
RE_NOTA_HASTA_FIN = re.compile(
    r'(?:Texto subrayado|Parágrafo declarado|Inciso declarado|Declarado\s|NOTA:|La expresión subrayada).*$')
RE_NOTA_CAMBIO = re.compile(
    r'(?i)\b(?:modificad[oa]s?(?:\s+parcialmente)?|adicionad[oa]s?|sustituid[oa]|derogad[oa]|eliminad[oa]|reglamentad[oa])'
    r'\s+por\b.{0,90}?\b(?:ley|decreto|resoluci[oó]n)\b.{0,45}?\d+\s+de\s+\d{4}'
    r'(?:\s+Congreso de la República)?\.?(?:\s*El nuevo texto es el siguiente:)?')
RE_NOTA_CORCHETES = re.compile(r'Modificado\s*\[?\s*Artículo\s+\d+\s+LEY\s+\d+\s+de\s+\d{4}\s*\]?')
RE_NOTA_VER = re.compile(
    r'\bVer\s+(?:Resoluci[oó]n|Concepto|Fallo|Sentencia|Ley|Decreto|los\s+arts?\.?|arts?\.).*?\bde\s+\d{4}\b\)?')
RE_NUEVO_TEXTO = re.compile(r'(?:El nuevo texto es el siguiente:|^INEXEQUIBLE\.)\s*')
RE_LITERAL_ELIMINADO = re.compile(r'^[A-Z]\.\d+\.?\s+Eliminad[oa]\b')
RE_LITERAL_VACIO = re.compile(r'^[A-Z](?:\.\d+)?\.?$')
RE_REFORMA = re.compile(
    r'(?i)(?:modificad|adicionad|sustituid|derogad|eliminad)[oa]s?\b.{0,90}?\b(ley|decreto)(?:\s+nacional|\s+ley)?\s+0*(\d+)\s+de\s+(\d{4})')
RE_SUBRAYADO_INEXEQUIBLE = re.compile(
    r'(?i)subrayad[oa]s?\s+(?:que\s+se\s+declaró|declarad[oa]s?|fue\s+declarad[oa])\s+INEXEQUIBLE')


def descargar_html(url=FUENTE_URL, timeout=120):
    der = requests.get(INTERMEDIO_URL, timeout=timeout).content
    ruta_bundle = Path(tempfile.gettempdir()) / 'bundle_funcionpublica.pem'
    ruta_bundle.write_text(Path(certifi.where()).read_text() + '\n' + ssl.DER_cert_to_PEM_cert(der))
    r = requests.get(url, timeout=timeout, verify=str(ruta_bundle), headers={'User-Agent': 'Mozilla/5.0'})
    r.raise_for_status()
    # La página declara ISO-8859-1 pero el contenido real es UTF-8.
    return r.content.decode('utf-8-sig')


def normalizar(texto):
    return ' '.join(texto.replace('\xa0', ' ').split())


def limpiar_parrafo(texto):
    if RE_PARRAFO_NOTA.match(texto):
        return ''
    texto = RE_NOTA_PARENTESIS.sub('', texto)
    texto = RE_NOTA_CORCHETES.sub('', texto)
    texto = RE_NOTA_HASTA_FIN.sub('', texto)
    texto = RE_NOTA_CAMBIO.sub('', texto)
    texto = RE_NOTA_VER.sub('', texto)
    texto = re.sub(r'\s+([,.;:])', r'\1', texto)
    texto = re.sub(r'^[\s,.;:]+', '', texto)
    texto = RE_NUEVO_TEXTO.sub('', texto)
    if '?' not in texto:
        texto = texto.replace('¿', '')
    texto = re.sub(r'([.;:,])(?:\s*[.,])+', r'\1', texto)
    return normalizar(texto)


def extraer_nombre(texto):
    m = re.match(r'^(.+?)\.(?:\s|$)', texto)
    if not m:
        return '', texto
    candidato = m.group(1)
    letras = [c for c in candidato if c.isalpha()]
    mayusculas = sum(c.isupper() for c in letras) / max(len(letras), 1)
    palabras = len(candidato.split())
    if mayusculas > 0.7 or palabras <= 8 or (palabras <= 16 and ',' not in candidato):
        return candidato.strip(), texto[m.end():].strip()
    return '', texto


def parsear_ley(html, fuente_url=FUENTE_URL):
    sopa = BeautifulSoup(html, 'html.parser')
    contenido = sopa.select_one('div.descripcion-contenido')
    for boton in contenido.find_all('button'):
        boton.decompose()
    for oculto in contenido.select('div[id^=juris]'):
        oculto.decompose()

    articulos, actual = [], None
    titulo = titulo_nombre = capitulo = capitulo_nombre = ''
    esperando = None
    omitiendo = False
    for bloque in contenido.find_all(['p', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6']):
        texto = normalizar(bloque.get_text(' '))
        if not texto:
            continue
        if RE_FIN.match(texto):
            break
        m_tit, m_cap = RE_TITULO.match(texto), RE_CAPITULO.match(texto)
        m_art, m_tra = RE_ARTICULO.match(texto), RE_TRANSITORIO.match(texto)
        if m_tit:
            titulo, titulo_nombre, capitulo, capitulo_nombre = m_tit.group(1).replace(' ', ''), '', '', ''
            actual, esperando, omitiendo = None, 'titulo', False
            continue
        if m_cap:
            capitulo, capitulo_nombre = m_cap.group(1), limpiar_parrafo(m_cap.group(2)).rstrip('.')
            actual, esperando, omitiendo = None, None if capitulo_nombre else 'capitulo', False
            continue
        if m_art or m_tra:
            if m_art:
                numero = m_art.group(1) + (m_art.group(2) or '') + (f'-{m_art.group(3)}' if m_art.group(3) else '')
            else:
                numero = 'Transitorio'
            actual = {'articulo': numero, 'titulo': titulo, 'titulo_nombre': titulo_nombre,
                      'capitulo': capitulo, 'capitulo_nombre': capitulo_nombre, 'bloques': [bloque]}
            articulos.append(actual)
            esperando, omitiendo = None, False
            continue
        if esperando == 'titulo':
            titulo_nombre = limpiar_parrafo(texto).rstrip('.')
            esperando = None
            continue
        if esperando == 'capitulo':
            capitulo_nombre = limpiar_parrafo(texto).rstrip('.')
            esperando = None
            continue
        if actual is None or omitiendo:
            continue
        if RE_OMITIR_DESDE.match(texto):
            omitiendo = True
            continue
        actual['bloques'].append(bloque)

    registros = []
    for orden, art in enumerate(articulos):
        crudo = '\n'.join(normalizar(b.get_text(' ')) for b in art['bloques'])
        if RE_SUBRAYADO_INEXEQUIBLE.search(crudo):
            for bloque in art['bloques']:
                for subrayado in bloque.find_all('u'):
                    subrayado.decompose()
        parrafos = [normalizar(b.get_text(' ')) for b in art['bloques']]
        encabezado = RE_ARTICULO.match(parrafos[0]) or RE_TRANSITORIO.match(parrafos[0])
        resto = limpiar_parrafo(encabezado.groups()[-1])
        # Literales eliminados por reformas posteriores (p. ej. E.3 del art. 131).
        cuerpo = [limpiar_parrafo(p) for p in parrafos[1:] if not RE_LITERAL_ELIMINADO.match(p)]
        cuerpo = [p for p in cuerpo if p and not RE_LITERAL_VACIO.match(p)]
        if resto:
            nombre, resto = extraer_nombre(resto)
        elif cuerpo:
            nombre, primero = extraer_nombre(cuerpo[0])
            cuerpo = ([primero] if primero else []) + cuerpo[1:]
        else:
            nombre = ''
        texto = '\n'.join(([resto] if resto else []) + cuerpo)
        reformas = sorted({f'{t.capitalize()} {int(n)} de {a}' for t, n, a in RE_REFORMA.findall(crudo)
                           if not (n.lstrip('0') == '769' and a == '2002')},
                          key=lambda s: (int(s.split()[-1]), s))
        encabezado_crudo = parrafos[0][:200]
        vigente = bool(texto) and not (
            re.search(r'INEXEQUIBLE\.', encabezado_crudo) and 'subrayad' not in encabezado_crudo
        ) and 'Este artículo fue derogado' not in crudo
        registros.append({
            'orden': orden, 'articulo': art['articulo'], 'nombre': nombre.rstrip('.').strip(),
            'titulo': art['titulo'], 'titulo_nombre': art['titulo_nombre'],
            'capitulo': art['capitulo'], 'capitulo_nombre': art['capitulo_nombre'],
            'texto': texto, 'modificado_por': reformas, 'vigente': vigente, 'fuente_url': fuente_url,
        })
    return registros

In [ ]:
RUTA_LOCAL = Path('../data')
URL_REPO = 'https://raw.githubusercontent.com/777wills/miaa-nlp/main/data'
ARCHIVO_LEY = 'ley_769_2002_articulos.jsonl'
ARCHIVO_PREGUNTAS = 'preguntas_evaluacion_transito.csv'


def leer_recurso(nombre):
    ruta = RUTA_LOCAL / nombre
    if ruta.exists():
        return ruta.read_bytes(), f'archivo local {ruta}'
    try:
        r = requests.get(f'{URL_REPO}/{nombre}', timeout=60)
        r.raise_for_status()
        return r.content, 'snapshot del repositorio en GitHub'
    except requests.RequestException as error:
        print(f'No se pudo leer {nombre} del repositorio: {error}')
        return None, None


contenido_ley, origen_ley = leer_recurso(ARCHIVO_LEY)
if contenido_ley is None:
    registros = parsear_ley(descargar_html())
    for registro in registros:
        registro['fecha_descarga'] = date.today().isoformat()
    contenido_ley = '\n'.join(json.dumps(r, ensure_ascii=False) for r in registros).encode('utf-8')
    origen_ley = 'descarga en vivo desde Función Pública'
else:
    registros = [json.loads(linea) for linea in contenido_ley.decode('utf-8').splitlines() if linea.strip()]

ley = pd.DataFrame(registros)
print('Origen:', origen_ley)
print('SHA-256:', hashlib.sha256(contenido_ley).hexdigest())
print('Fecha de descarga del texto:', ley['fecha_descarga'].iloc[0])
print(f'Artículos: {len(ley)} | vigentes: {ley.vigente.sum()} | no vigentes: {(~ley.vigente).sum()}')
ley[['articulo', 'nombre', 'titulo', 'capitulo', 'modificado_por', 'vigente']].head(8)

Para ver qué tanto cambia el texto con la limpieza, descargamos la página en vivo y comparamos el artículo 131 (multas) tal como viene en el HTML, incluidas las versiones ocultas, con su versión limpia. También verificamos si la página cambió desde que tomamos el snapshot. Si la página no responde, la celda lo informa y el resto del cuaderno sigue funcionando con el snapshot.

In [ ]:
try:
    html_vivo = descargar_html()
except requests.RequestException as error:
    html_vivo = None
    print('La página oficial no respondió; seguimos con el snapshot.', error)

if html_vivo is not None:
    contenido_vivo = BeautifulSoup(html_vivo, 'html.parser').select_one('div.descripcion-contenido')
    texto_crudo = normalizar(contenido_vivo.get_text(' '))
    inicio, fin = texto_crudo.find('ARTÍCULO 131.'), texto_crudo.find('ARTÍCULO 132.')
    crudo_131 = texto_crudo[inicio:fin]
    limpio_131 = ley.loc[ley.articulo == '131', 'texto'].iloc[0]

    print(f'Página completa: {len(texto_crudo.split()):,} palabras en crudo '
          f'frente a {ley.texto.str.split().str.len().sum():,} después de limpiar')
    print(f'Artículo 131: {len(crudo_131.split()):,} palabras en crudo frente a {len(limpio_131.split()):,} limpio')
    texto_limpio = '\n'.join(ley.texto)
    for marca in ['Norma Anterior', 'Texto anterior', 'Modificado por', 'EXEQUIBLE', 'Ver Resolución']:
        print(f'  "{marca}": {texto_crudo.count(marca)} veces en la página, {texto_limpio.count(marca)} en el corpus limpio')
    print('\nInicio en crudo:\n', crudo_131[:500])
    print('\nInicio limpio:\n', limpio_131[:500])

    vivo = pd.DataFrame(parsear_ley(html_vivo))
    comparacion = ley[['orden', 'articulo', 'texto']].merge(vivo[['orden', 'articulo', 'texto']], on=['orden', 'articulo'],
                                                             how='outer', suffixes=('_snapshot', '_vivo'), indicator=True)
    cambiados = comparacion[(comparacion['_merge'] != 'both') |
                            (comparacion['texto_snapshot'] != comparacion['texto_vivo'])]
    print(f'\nArtículos que cambiaron desde el snapshot: {len(cambiados)}')

---
## 5. Análisis exploratorio del corpus

Antes de decidir cómo partir el texto queremos saber cómo está repartido. Nos interesan tres cosas: cómo se distribuyen los artículos entre los títulos del código, qué tan largos son (eso define el tamaño de los fragmentos) y qué tanto ha sido reformada la ley.

In [ ]:
ley['palabras'] = ley['texto'].str.split().str.len()
vigentes = ley[ley.vigente].reset_index(drop=True)

print('Artículos excluidos del índice:')
display(ley.loc[~ley.vigente, ['articulo', 'nombre', 'palabras', 'modificado_por']])

por_titulo = (vigentes.assign(titulo_completo=vigentes.titulo + ' · ' + vigentes.titulo_nombre.str.capitalize())
              .groupby('titulo_completo', sort=False)
              .agg(articulos=('articulo', 'count'), palabras=('palabras', 'sum')))

fig, ejes = plt.subplots(1, 2, figsize=(13, 3.8))
por_titulo['articulos'].plot.barh(ax=ejes[0], color='#4C72B0')
ejes[0].set_title('Artículos vigentes por título')
ejes[0].invert_yaxis()
por_titulo['palabras'].plot.barh(ax=ejes[1], color='#DD8452')
ejes[1].set_title('Palabras por título')
ejes[1].invert_yaxis()
ejes[1].set_ylabel('')
plt.tight_layout()
plt.show()

### Longitud de los artículos

Esta es la variable que más nos importa para fragmentar. Si la mayoría de artículos son cortos, un artículo puede ser un fragmento natural; si hay artículos muy largos, un solo vector tendría que resumir muchos temas a la vez.

In [ ]:
percentiles = vigentes['palabras'].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(0)
print(vigentes['palabras'].describe().round(1).to_string())
print('\nPercentiles:\n', percentiles.to_string())

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
ejes[0].hist(vigentes['palabras'], bins=40, color='#4C72B0')
for q, estilo in [(0.50, '--'), (0.75, ':'), (0.90, '-.')]:
    valor = vigentes['palabras'].quantile(q)
    ejes[0].axvline(valor, color='black', linestyle=estilo, label=f'p{int(q * 100)} = {valor:.0f}')
ejes[0].set_title('Distribución de palabras por artículo')
ejes[0].set_xlabel('Palabras')
ejes[0].set_ylabel('Artículos')
ejes[0].legend()
ejes[1].hist(vigentes['palabras'], bins=np.logspace(np.log10(5), np.log10(vigentes['palabras'].max()), 30), color='#55A868')
ejes[1].set_xscale('log')
ejes[1].set_title('La misma distribución en escala logarítmica')
ejes[1].set_xlabel('Palabras (escala log)')
plt.tight_layout()
plt.show()

print('Artículos más largos:')
vigentes.nlargest(8, 'palabras')[['articulo', 'nombre', 'palabras']]

### Reformas de la ley

El campo `modificado_por` guarda las leyes y decretos que tocaron cada artículo. Nos sirve para entender por qué el texto es difícil de seguir para un ciudadano, y para identificar los temas que más se han reformado.

In [ ]:
reformas = vigentes[['articulo', 'modificado_por']].explode('modificado_por').dropna()
reformas['anio'] = reformas['modificado_por'].str.extract(r'(\d{4})$').astype(int)

print(f'Artículos vigentes con al menos una reforma: {(vigentes.modificado_por.apply(len) > 0).sum()} '
      f'de {len(vigentes)}')
print(f'Normas distintas que modificaron el código: {reformas.modificado_por.nunique()}')

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
reformas['modificado_por'].value_counts().head(12).sort_values().plot.barh(ax=ejes[0], color='#8172B3')
ejes[0].set_title('Normas que más artículos modificaron')
ejes[0].set_xlabel('Artículos modificados')
reformas.groupby('anio').size().plot.bar(ax=ejes[1], color='#C44E52')
ejes[1].set_title('Artículos modificados por año de la norma')
ejes[1].set_xlabel('Año')
plt.tight_layout()
plt.show()

vigentes.assign(reformas=vigentes.modificado_por.apply(len)).nlargest(6, 'reformas')[['articulo', 'nombre', 'reformas']]

---
## 6. Fragmentación

Comparamos tres formas de partir los 176 artículos vigentes:

- **(a) Artículo completo.** Cada artículo es un fragmento. Es la opción más simple, pero un artículo como el 131 (multas) o el 2 (definiciones) queda en un solo vector que mezcla decenas de temas. Además, los modelos E5 solo leen 512 tokens, así que el resto del artículo ni siquiera entra al embedding.
- **(b) Ventana deslizante de palabras.** Fragmentos de tamaño fijo con solapamiento. Elegimos **100 palabras con paso de 50** porque la mediana de los artículos está alrededor de 94 palabras: así la mitad de los artículos cabe completa en una ventana, y el solapamiento del 50 % evita que una idea quede partida justo en el borde.
- **(c) Estructural.** Los artículos de hasta 150 palabras (cerca del 70 % del corpus) se dejan enteros. Los más largos se parten por su propia estructura: cada parágrafo inicia un fragmento nuevo, y cuando un párrafo termina en dos puntos (por ejemplo, "Será sancionado con multa equivalente a quince (15) SMLDV… las siguientes infracciones:") lo repetimos al inicio de cada fragmento de la lista que sigue. Así un fragmento con el literal C.2 lleva también el valor de la multa. Los elementos de una lista se agrupan hasta llegar a 150 palabras.

Para que la comparación sea justa, las tres estrategias anteponen el mismo encabezado a cada fragmento (número y nombre del artículo, título y capítulo). Ese encabezado también le sirve al modelo generador para saber qué artículo está leyendo y citarlo.

In [ ]:
from langchain_core.documents import Document

TAM_VENTANA, PASO_VENTANA = 100, 50
MAX_PALABRAS_ESTRUCTURAL = 150
RE_PARAGRAFO = re.compile(r'^P[AÁ]R[AÁ]GRAFO', re.IGNORECASE)


def contar_palabras(texto):
    return len(texto.split())


def encabezado(fila):
    ubicacion = ' · '.join(x for x in [f'Título {fila.titulo} {fila.titulo_nombre.capitalize()}'.strip(),
                                       f'Capítulo {fila.capitulo} {fila.capitulo_nombre.capitalize()}'.strip()
                                       if fila.capitulo else ''] if x)
    nombre = f' – {fila.nombre}' if fila.nombre else ''
    return f'Artículo {fila.articulo}{nombre} (Ley 769 de 2002; {ubicacion})'


def documento(fila, texto, numero):
    return Document(page_content=f'{encabezado(fila)}\n{texto}',
                    metadata={'articulo': fila.articulo, 'nombre': fila.nombre, 'titulo': fila.titulo,
                              'capitulo': fila.capitulo, 'fragmento': numero})


def fragmentar_articulo(fila):
    return [documento(fila, fila.texto, 0)]


def fragmentar_ventana(fila, tam=TAM_VENTANA, paso=PASO_VENTANA):
    palabras = fila.texto.split()
    inicios = range(0, max(len(palabras) - tam + paso, 1), paso)
    return [documento(fila, ' '.join(palabras[i:i + tam]), n) for n, i in enumerate(inicios)]


def fragmentar_estructural(fila, max_palabras=MAX_PALABRAS_ESTRUCTURAL):
    if fila.palabras <= max_palabras:
        return [documento(fila, fila.texto, 0)]
    fragmentos, actual, introduccion = [], [], ''

    def cerrar():
        if actual:
            fragmentos.append('\n'.join(([introduccion] if introduccion else []) + actual))
            actual.clear()

    for parrafo in fila.texto.split('\n'):
        if RE_PARAGRAFO.match(parrafo):
            cerrar()
            introduccion = ''
        elif parrafo.endswith(':'):
            cerrar()
            introduccion = parrafo
            continue
        elif actual and sum(map(contar_palabras, actual)) + contar_palabras(parrafo) > max_palabras:
            cerrar()
        actual.append(parrafo)
    cerrar()
    if not fragmentos and introduccion:
        fragmentos.append(introduccion)
    return [documento(fila, texto, n) for n, texto in enumerate(fragmentos)]


ESTRATEGIAS = {'articulo': fragmentar_articulo, 'ventana': fragmentar_ventana, 'estructural': fragmentar_estructural}
FRAGMENTOS = {nombre: [doc for fila in vigentes.itertuples() for doc in funcion(fila)]
              for nombre, funcion in ESTRATEGIAS.items()}

resumen_fragmentos = pd.DataFrame({
    nombre: pd.Series([contar_palabras(d.page_content) for d in docs]).describe()[['count', 'mean', '50%', 'max']]
    for nombre, docs in FRAGMENTOS.items()}).T.round(1)
resumen_fragmentos.columns = ['fragmentos', 'palabras media', 'palabras mediana', 'palabras máx.']
resumen_fragmentos

In [ ]:
fig, eje = plt.subplots(figsize=(9, 3.8))
eje.boxplot([[contar_palabras(d.page_content) for d in docs] for docs in FRAGMENTOS.values()], vert=False)
eje.set_yticks(range(1, len(FRAGMENTOS) + 1), list(FRAGMENTOS))
eje.set_xscale('log')
eje.set_xlabel('Palabras por fragmento (escala log, incluye el encabezado)')
eje.set_title('Longitud de los fragmentos según la estrategia')
plt.tight_layout()
plt.show()

ejemplo = [d for d in FRAGMENTOS['estructural'] if d.metadata['articulo'] == '131']
print(f'El artículo 131 queda en {len(ejemplo)} fragmentos estructurales. Uno de ellos:\n')
print(next(d.page_content for d in ejemplo if 'C.2.' in d.page_content)[:900])

---
## 7. Set de evaluación

Para comparar configuraciones necesitamos preguntas con su respuesta conocida. Escribimos **46 preguntas** como las haría cualquier persona ("¿me pueden multar por hablar por celular mientras manejo?") y, para cada una, anotamos el artículo o artículos donde está la respuesta, verificándolo contra el texto limpio. Las preguntas cubren licencias, documentos del vehículo, normas de comportamiento, motos, bicicletas, peatones y sanciones. Agregamos **6 preguntas que el código no responde** (impuestos de una ciudad, precio de la gasolina, recetas de cocina…) para medir si el asistente sabe decir que no tiene la información.

Con este set medimos la recuperación a nivel de artículo:

- **Recall@k:** qué fracción de los artículos correctos aparece entre los k fragmentos recuperados.
- **MRR@10:** el promedio de 1/posición del primer fragmento que pertenece a un artículo correcto. Vale 1 si el primero ya es correcto y 0,5 si es el segundo.

In [ ]:
contenido_preguntas, origen_preguntas = leer_recurso(ARCHIVO_PREGUNTAS)
if contenido_preguntas is None:
    raise RuntimeError('El set de evaluación solo está en el repositorio; revisa la conexión a GitHub.')
preguntas = pd.read_csv(io.BytesIO(contenido_preguntas), dtype={'articulos_gold': str})
preguntas['articulos_gold'] = preguntas['articulos_gold'].fillna('').apply(lambda s: set(filter(None, s.split(';'))))
preguntas['en_dominio'] = preguntas['en_dominio'].astype(bool)
en_dominio = preguntas[preguntas.en_dominio].reset_index(drop=True)
fuera_dominio = preguntas[~preguntas.en_dominio].reset_index(drop=True)

articulos_vigentes = set(vigentes.articulo)
faltantes = {a for gold in en_dominio.articulos_gold for a in gold} - articulos_vigentes
assert not faltantes, f'Artículos gold que no están en el índice: {faltantes}'

print('Origen:', origen_preguntas)
print(f'Preguntas: {len(preguntas)} ({len(en_dominio)} del código, {len(fuera_dominio)} por fuera)')
print(preguntas['tema'].value_counts().to_string())
preguntas.sample(6, random_state=SEED)[['pregunta', 'articulos_gold', 'tema']]

Una forma de anticipar qué tan difícil es la recuperación es medir cuántas palabras de cada pregunta aparecen literalmente en su artículo correcto. Si el solapamiento es bajo, una búsqueda por palabras clave fallaría y necesitamos embeddings que capturen el significado ("pase" por "licencia de conducción", "parquear" por "estacionar").

In [ ]:
VACIAS = set("""
a al algo algún alguna como con cual cuál cuales cuando cuándo cuánto cuánta cuántos de del desde donde dónde el
en es esa ese eso esta este esto hay la las lo los me mi mis más o para pero por que qué quien quién se si sí sin
su sus también tengo tiene un una uno y ya yo le les muy puedo pueden debo hacer hago son está están
""".split())


def palabras_contenido(texto):
    sin_tildes = unicodedata.normalize('NFKD', texto.lower()).encode('ascii', 'ignore').decode()
    return {p for p in re.findall(r'[a-zñ]+', sin_tildes) if len(p) > 2 and p not in VACIAS}


texto_por_articulo = dict(zip(vigentes.articulo, vigentes.texto + ' ' + vigentes.nombre))
en_dominio['solapamiento'] = [
    len(palabras_contenido(q) & palabras_contenido(' '.join(texto_por_articulo[a] for a in gold))) /
    max(len(palabras_contenido(q)), 1)
    for q, gold in zip(en_dominio.pregunta, en_dominio.articulos_gold)]

print(f'Solapamiento léxico medio: {en_dominio.solapamiento.mean():.1%}')
fig, eje = plt.subplots(figsize=(9, 3.5))
en_dominio.boxplot(column='solapamiento', by='tema', ax=eje, vert=False)
eje.set_title('Fracción de palabras de la pregunta presentes en el artículo correcto')
eje.set_xlabel('Solapamiento')
plt.suptitle('')
plt.tight_layout()
plt.show()
en_dominio.nsmallest(5, 'solapamiento')[['pregunta', 'articulos_gold', 'solapamiento']]

---
## 8. Experimentos de recuperación

Combinamos las tres estrategias de fragmentación con cuatro variantes de embeddings:

| Variante | Modelo | Por qué la incluimos |
|---|---|---|
| e5-large sin prefijos | `intfloat/multilingual-e5-large` | Uso directo del modelo, sin ninguna instrucción adicional |
| e5-large con prefijos | `intfloat/multilingual-e5-large` | La forma recomendada por sus autores: `query:` en la pregunta y `passage:` en el texto |
| e5-base con prefijos | `intfloat/multilingual-e5-base` | La mitad de parámetros: ¿cuánto se pierde con un modelo más liviano? |
| bge-m3 | `BAAI/bge-m3` | Modelo multilingüe más reciente, que admite textos largos. Lo limitamos a 1.024 tokens para no quedarnos sin memoria en la T4 |

Cada combinación se indexa en FAISS con producto interno sobre vectores normalizados (equivale a similitud coseno). Para cada pregunta recuperamos 20 candidatos y los evaluamos de dos formas: en el orden que da FAISS, y reordenados con el cross-encoder `BAAI/bge-reranker-v2-m3`, que lee la pregunta y el fragmento juntos en lugar de comparar dos vectores independientes. Guardamos los 20 candidatos con sus puntajes, así podemos calcular las métricas para cualquier k sin repetir búsquedas, y también reutilizarlos después para calibrar el umbral de abstención.

Además del acierto medimos el **costo**: cuántas palabras de contexto recibe el modelo generador con k = 5. Una estrategia que acierta igual pero entrega la mitad del texto es preferible, porque los modelos pequeños se confunden con contextos largos.

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_huggingface import HuggingFaceEmbeddings

MODELOS_EMBEDDING = {
    'e5-large sin prefijos': {'modelo': 'intfloat/multilingual-e5-large', 'consulta': '', 'pasaje': ''},
    'e5-large con prefijos': {'modelo': 'intfloat/multilingual-e5-large', 'consulta': 'query: ', 'pasaje': 'passage: '},
    'e5-base con prefijos': {'modelo': 'intfloat/multilingual-e5-base', 'consulta': 'query: ', 'pasaje': 'passage: '},
    'bge-m3': {'modelo': 'BAAI/bge-m3', 'consulta': '', 'pasaje': '', 'max_tokens': 1024},
}
RERANKER_MODELO = 'BAAI/bge-reranker-v2-m3'
K_CANDIDATOS = 20
KS = (1, 3, 5, 10)


def crear_embeddings(cfg):
    base = {'normalize_embeddings': True, 'batch_size': 16}
    embeddings = HuggingFaceEmbeddings(
        model_name=cfg['modelo'], model_kwargs={'device': DEVICE},
        encode_kwargs={**base, **({'prompt': cfg['pasaje']} if cfg['pasaje'] else {})},
        query_encode_kwargs={**base, **({'prompt': cfg['consulta']} if cfg['consulta'] else {})})
    if 'max_tokens' in cfg:
        embeddings._client.max_seq_length = cfg['max_tokens']
    return embeddings


def crear_indice(documentos, embeddings):
    return FAISS.from_documents(documentos, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)


def liberar_memoria():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


reranker = HuggingFaceCrossEncoder(model_name=RERANKER_MODELO, model_kwargs={'device': DEVICE, 'max_length': 512})
print('Re-ranker cargado:', RERANKER_MODELO)

In [ ]:
candidatos, tiempos = [], []
for nombre_emb, cfg in MODELOS_EMBEDDING.items():
    inicio = time.time()
    embeddings = crear_embeddings(cfg)
    carga = time.time() - inicio
    for nombre_frag, documentos in FRAGMENTOS.items():
        inicio = time.time()
        indice = crear_indice(documentos, embeddings)
        indexacion = time.time() - inicio
        inicio = time.time()
        for fila in preguntas.itertuples():
            resultados = indice.similarity_search_with_score(fila.pregunta, k=K_CANDIDATOS)
            puntajes = reranker.score([(fila.pregunta, doc.page_content) for doc, _ in resultados])
            for posicion, ((doc, similitud), puntaje) in enumerate(zip(resultados, puntajes)):
                candidatos.append({'embedding': nombre_emb, 'estrategia': nombre_frag, 'id': fila.id,
                                   'posicion': posicion, 'articulo': doc.metadata['articulo'],
                                   'similitud': float(similitud), 'puntaje_reranker': float(puntaje),
                                   'palabras': contar_palabras(doc.page_content)})
        tiempos.append({'embedding': nombre_emb, 'estrategia': nombre_frag, 'carga_modelo_s': carga,
                        'indexacion_s': indexacion, 'busqueda_y_reranking_s': time.time() - inicio})
        print(f'{nombre_emb:22s} | {nombre_frag:11s} | {len(documentos):4d} fragmentos | '
              f'indexación {indexacion:5.1f} s')
    del embeddings, indice
    liberar_memoria()

candidatos = pd.DataFrame(candidatos)
pd.DataFrame(tiempos).round(1)

Con los candidatos guardados calculamos las métricas. Para la versión con re-ranking ordenamos los 20 candidatos de cada pregunta por el puntaje del cross-encoder; sin re-ranking conservamos el orden de FAISS.

In [ ]:
gold_por_id = dict(zip(preguntas.id, preguntas.articulos_gold))
ids_en_dominio = set(en_dominio.id)


def metricas_ranking(articulos, gold):
    fila = {f'recall@{k}': len(set(articulos[:k]) & gold) / len(gold) for k in KS}
    fila['mrr@10'] = next((1 / (i + 1) for i, a in enumerate(articulos[:10]) if a in gold), 0.0)
    return fila


filas = []
for (nombre_emb, nombre_frag, qid), grupo in candidatos[candidatos.id.isin(ids_en_dominio)].groupby(
        ['embedding', 'estrategia', 'id'], sort=False):
    for con_reranker in (False, True):
        orden = grupo.sort_values('puntaje_reranker' if con_reranker else 'posicion', ascending=not con_reranker)
        fila = metricas_ranking(orden['articulo'].tolist(), gold_por_id[qid])
        fila.update({'embedding': nombre_emb, 'estrategia': nombre_frag, 'reranker': con_reranker, 'id': qid,
                     'palabras_contexto@5': orden['palabras'].head(5).sum()})
        filas.append(fila)

por_pregunta = pd.DataFrame(filas)
tabla_recuperacion = (por_pregunta.groupby(['embedding', 'estrategia', 'reranker'], sort=False)
                      .mean(numeric_only=True).drop(columns='id').round(3))
tabla_recuperacion.sort_values(['recall@5', 'mrr@10'], ascending=False).head(12)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for eje, con_reranker in zip(ejes, (False, True)):
    matriz = tabla_recuperacion.xs(con_reranker, level='reranker')['recall@5'].unstack('estrategia')
    matriz = matriz.loc[list(MODELOS_EMBEDDING), list(ESTRATEGIAS)]
    imagen = eje.imshow(matriz.values, cmap='viridis', vmin=0, vmax=1)
    eje.set_xticks(range(matriz.shape[1]), matriz.columns)
    eje.set_yticks(range(matriz.shape[0]), matriz.index)
    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            eje.text(j, i, f'{matriz.values[i, j]:.2f}', ha='center', va='center',
                     color='white' if matriz.values[i, j] < 0.6 else 'black')
    eje.set_title(f'Recall@5 {"con" if con_reranker else "sin"} re-ranking')
fig.colorbar(imagen, ax=ejes, shrink=0.8)
plt.show()

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

mejores = tabla_recuperacion.sort_values(['recall@5', 'mrr@10'], ascending=False).head(4)
for (nombre_emb, nombre_frag, con_reranker), fila in mejores.iterrows():
    etiqueta = f'{nombre_emb} · {nombre_frag}{" · rerank" if con_reranker else ""}'
    ejes[0].plot(KS, [fila[f'recall@{k}'] for k in KS], marker='o', label=etiqueta)
ejes[0].set_xticks(KS)
ejes[0].set_xlabel('k (fragmentos recuperados)')
ejes[0].set_ylabel('Recall@k')
ejes[0].set_title('Recall según k: las 4 mejores configuraciones')
ejes[0].legend(fontsize=7)

prefijos = (tabla_recuperacion.xs(False, level='reranker')
            .loc[['e5-large sin prefijos', 'e5-large con prefijos'], 'recall@5'].unstack('embedding'))
prefijos.loc[list(ESTRATEGIAS)].plot.bar(ax=ejes[1], rot=0)
ejes[1].set_title('Efecto de los prefijos de E5 (Recall@5, sin re-ranking)')
ejes[1].set_ylim(0, 1)

efecto = tabla_recuperacion['mrr@10'].unstack('reranker')
efecto.columns = ['sin re-ranking', 'con re-ranking']
efecto.groupby(level='estrategia').mean().loc[list(ESTRATEGIAS)].plot.bar(ax=ejes[2], rot=0)
ejes[2].set_title('MRR@10 promedio por estrategia')
ejes[2].set_ylim(0, 1)
plt.tight_layout()
plt.show()

(tabla_recuperacion.groupby(level=['estrategia', 'reranker'])[['recall@5', 'mrr@10', 'palabras_contexto@5']]
 .mean().round(3).loc[list(ESTRATEGIAS)])

### Selección de la configuración final

Elegimos la configuración con mayor Recall@5 y, si hay empate, la de mayor MRR@10. Después fijamos k con una regla sencilla: el menor k cuyo recall llega al 95 % del Recall@10 de esa configuración. Así no le pasamos al modelo generador más contexto del necesario.

In [ ]:
(EMB_FINAL, ESTRATEGIA_FINAL, RERANKER_FINAL), fila_final = next(
    tabla_recuperacion.sort_values(['recall@5', 'mrr@10'], ascending=False).iterrows())
K_FINAL = next(k for k in KS if fila_final[f'recall@{k}'] >= 0.95 * fila_final['recall@10'])

print('Embedding:', EMB_FINAL)
print('Estrategia de fragmentación:', ESTRATEGIA_FINAL)
print('Re-ranking:', 'sí' if RERANKER_FINAL else 'no')
print('k:', K_FINAL)
print(fila_final.round(3).to_string())

fallos = por_pregunta[(por_pregunta.embedding == EMB_FINAL) & (por_pregunta.estrategia == ESTRATEGIA_FINAL) &
                      (por_pregunta.reranker == RERANKER_FINAL) & (por_pregunta[f'recall@{K_FINAL}'] < 1)]
print(f'\nPreguntas en las que no se recuperan todos los artículos correctos con k = {K_FINAL}: {len(fallos)}')
fallos.merge(preguntas[['id', 'pregunta', 'articulos_gold']], on='id')[
    ['pregunta', 'articulos_gold', f'recall@{K_FINAL}', 'mrr@10']]

---
## 9. Modelos generadores con Ollama

Servimos los modelos de lenguaje con Ollama dentro de la misma máquina. Como no tenemos una terminal a mano, instalamos Ollama si hace falta, arrancamos el servidor en segundo plano con `subprocess` y esperamos a que responda en el puerto 11434 antes de continuar.

Comparamos dos modelos pequeños que caben sin problema en una T4: `llama3.2:3b` (Meta) y `gemma3:4b` (Google). Ambos trabajan con temperatura 0 y semilla fija para que las respuestas sean repetibles, con una ventana de contexto de 8.192 tokens para que los fragmentos no se trunquen y con un máximo de 512 tokens por respuesta, para que un modelo que entre en un bucle no bloquee la ejecución.

In [ ]:
def ollama_activo(host='127.0.0.1', puerto=11434):
    with socket.socket() as conexion:
        conexion.settimeout(1)
        return conexion.connect_ex((host, puerto)) == 0


if shutil.which('ollama') is None:
    if not IN_COLAB:
        raise RuntimeError('Instala Ollama desde https://ollama.com/download y vuelve a ejecutar la celda.')
    # El instalador de Ollama necesita zstd para descomprimir el binario.
    !sudo apt-get install -y -qq zstd > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh

if not ollama_activo():
    servidor_ollama = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):
        if ollama_activo():
            break
        time.sleep(1)
print('Servidor de Ollama activo:', ollama_activo())

In [ ]:
from langchain_ollama import ChatOllama

LLMS = ['llama3.2:3b', 'gemma3:4b']
for modelo in LLMS:
    inicio = time.time()
    subprocess.run(['ollama', 'pull', modelo], check=True, capture_output=True)
    print(f'{modelo} listo en {time.time() - inicio:.0f} s')


def crear_llm(modelo):
    return ChatOllama(model=modelo, temperature=0, seed=SEED, num_ctx=8192, num_predict=512,
                      validate_model_on_init=True)


def descargar_de_memoria(modelo):
    subprocess.run(['ollama', 'stop', modelo], capture_output=True)

Antes de conectar la recuperación, vale la pena ver qué responden los modelos **sin contexto**, solo con lo que aprendieron en su entrenamiento. Les pedimos además que citen el artículo, para ver si lo inventan.

In [ ]:
PREGUNTAS_SIN_CONTEXTO = [16, 35, 38]
for modelo in LLMS:
    llm = crear_llm(modelo)
    print('=' * 100, f'\n{modelo}')
    for qid in PREGUNTAS_SIN_CONTEXTO:
        pregunta = preguntas.loc[preguntas.id == qid, 'pregunta'].iloc[0]
        respuesta = llm.invoke(f'{pregunta} Responde en máximo tres frases y cita el artículo de la Ley 769 de '
                               f'2002 (Código Nacional de Tránsito de Colombia) que lo establece.').content
        print(f'\n• {pregunta}  [artículos correctos: {sorted(gold_por_id[qid])}]\n{respuesta}')
    descargar_de_memoria(modelo)

---
## 10. Cadena RAG con citas

Armamos la cadena con las piezas de LangChain `create_stuff_documents_chain` y `create_retrieval_chain`. Le hicimos tres cambios para este caso:

1. **Un recuperador propio (`RecuperadorLey`)** que busca en FAISS, reordena con el cross-encoder si la configuración elegida usa re-ranking, y guarda el puntaje de cada fragmento en sus metadatos (el del cross-encoder o, si no hay re-ranking, la similitud del embedding). Con ese puntaje mostramos la relevancia en la interfaz y aplicamos el umbral de abstención.
2. **Un atajo para cuando no hay contexto.** Si el recuperador no devuelve nada, porque ningún fragmento supera el umbral, la cadena responde un mensaje fijo sin llamar al modelo. Así el modelo no tiene oportunidad de inventar.
3. **Referencias armadas desde los metadatos.** Cada respuesta termina con la lista de artículos consultados, tomada de los metadatos de los fragmentos. No hace falta buscar el texto de vuelta en el corpus.

El prompt le pide al modelo que use solo el contexto, que cite con el formato "(Art. N)" y que, si no encuentra la respuesta, lo diga con una frase exacta que podamos detectar automáticamente.

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.prompts import PromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.runnables import RunnableBranch, RunnableLambda

MENSAJE_ABSTENCION = 'No encontré esa información en el Código Nacional de Tránsito.'


class RecuperadorLey(BaseRetriever):
    indice: Any
    reranker: Any = None
    k: int = 5
    k_candidatos: int = K_CANDIDATOS
    umbral: Optional[float] = None

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun):
        resultados = self.indice.similarity_search_with_score(query, k=self.k_candidatos if self.reranker else self.k)
        if self.reranker is not None:
            puntajes = self.reranker.score([(query, doc.page_content) for doc, _ in resultados])
            resultados = sorted(zip((doc for doc, _ in resultados), puntajes), key=lambda par: -par[1])
        documentos = [Document(page_content=doc.page_content, metadata={**doc.metadata, 'puntaje': float(p)})
                      for doc, p in resultados[:self.k]]
        if self.umbral is not None and (not documentos or documentos[0].metadata['puntaje'] < self.umbral):
            return []
        return documentos


PROMPT_QA = PromptTemplate.from_template(
    """Eres un asistente que orienta a ciudadanos colombianos sobre el Código Nacional de Tránsito (Ley 769 de 2002).
Responde en español, de forma clara y breve, usando únicamente la información de los artículos del contexto.
Cita siempre los artículos en los que te basas con el formato (Art. N).
No inventes valores, plazos ni artículos. Si el contexto no contiene la respuesta, responde exactamente:
"{abstencion}"

Contexto:
{context}

Pregunta: {input}
Respuesta:""").partial(abstencion=MENSAJE_ABSTENCION)


def con_atajo_de_abstencion(cadena_documentos):
    return RunnableBranch((lambda entrada: not entrada['context'], RunnableLambda(lambda _: MENSAJE_ABSTENCION)),
                          cadena_documentos)


def crear_cadena_qa(llm, recuperador, prompt=PROMPT_QA):
    return create_retrieval_chain(recuperador, con_atajo_de_abstencion(create_stuff_documents_chain(llm, prompt)))


def referencias(contexto):
    vistos = {}
    for doc in contexto:
        vistos.setdefault(doc.metadata['articulo'], doc.metadata['nombre'])
    return [f'Art. {a} – {n}' if n else f'Art. {a}' for a, n in vistos.items()]


def se_abstuvo(texto):
    return 'no encontré esa información' in texto.lower()


def formatear_respuesta(respuesta):
    texto = respuesta['answer'].strip()
    if respuesta['context'] and not se_abstuvo(texto):
        texto += '\n\n**Referencias (Ley 769 de 2002):**\n' + '\n'.join(
            f'- [{i}] {ref}' for i, ref in enumerate(referencias(respuesta['context']), 1))
    return texto

In [ ]:
embeddings_final = crear_embeddings(MODELOS_EMBEDDING[EMB_FINAL])
indice_final = crear_indice(FRAGMENTOS[ESTRATEGIA_FINAL], embeddings_final)
recuperador = RecuperadorLey(indice=indice_final, reranker=reranker if RERANKER_FINAL else None, k=K_FINAL)

llm = crear_llm(LLMS[0])
cadena_qa = crear_cadena_qa(llm, recuperador)
respuesta = cadena_qa.invoke({'input': '¿El parrillero de la moto también tiene que usar casco?'})
print(formatear_respuesta(respuesta))

---
## 11. Comparación de modelos generadores

Pasamos las 52 preguntas por la cadena con cada modelo, con la misma configuración de recuperación y todavía sin umbral. De cada respuesta extraemos los artículos citados con una expresión regular que reconoce formas como "(Art. 131)", "artículo 82" o "artículos 26 y 152", y calculamos:

- **Citas correctas:** preguntas del código en las que el modelo cita al menos un artículo correcto.
- **Respuestas con cita:** preguntas del código en las que el modelo cita algún artículo.
- **Citas fuera del contexto:** respuestas que citan un artículo que no estaba entre los fragmentos recuperados. Es la señal más clara de que el modelo se inventó la referencia.
- **Abstención fuera de dominio:** preguntas por fuera del código en las que el modelo dice que no encontró la información (lo deseable).
- **Abstención indebida:** preguntas del código en las que el modelo dice que no encontró la información aunque sí está.
- **Latencia media** por respuesta, sin contar la primera llamada, en la que se carga el modelo.

In [ ]:
RE_CITA = re.compile(r'\b[Aa]rt(?:[íi]culos?|s?\.)\s*((?:\d+(?:-\d+|\s?[A-Z](?![A-Za-záéíóúñ.\d]))?(?:\s*(?:,|y|e)\s*)?)+)')
RE_NUMERO_CITA = re.compile(r'\d+(?:-\d+|\s?[A-Z](?![A-Za-záéíóúñ.\d]))?')


def articulos_citados(texto):
    return {n.replace(' ', '') for grupo in RE_CITA.findall(texto) for n in RE_NUMERO_CITA.findall(grupo)}


print(articulos_citados('Según el (Art. 131 C.38) y los artículos 26 y 152, además del art. 85 A y el Artículo 93-2.'))

In [ ]:
respuestas_llm = []
for modelo in LLMS:
    cadena = crear_cadena_qa(crear_llm(modelo), recuperador)
    cadena.invoke({'input': preguntas.pregunta.iloc[0]})
    for fila in preguntas.itertuples():
        inicio = time.time()
        salida = cadena.invoke({'input': fila.pregunta})
        respuestas_llm.append({'modelo': modelo, 'id': fila.id, 'en_dominio': fila.en_dominio,
                               'respuesta': salida['answer'], 'latencia_s': time.time() - inicio,
                               'citados': articulos_citados(salida['answer']),
                               'contexto': {d.metadata['articulo'] for d in salida['context']},
                               'puntaje_max': max((d.metadata['puntaje'] for d in salida['context']), default=None)})
    descargar_de_memoria(modelo)
    print(f'{modelo}: {sum(r["modelo"] == modelo for r in respuestas_llm)} respuestas')

respuestas_llm = pd.DataFrame(respuestas_llm)
respuestas_llm['gold'] = respuestas_llm.id.map(gold_por_id)
respuestas_llm['abstencion'] = respuestas_llm.respuesta.apply(se_abstuvo)
respuestas_llm['cita_correcta'] = [bool(c & g) for c, g in zip(respuestas_llm.citados, respuestas_llm.gold)]
respuestas_llm['con_cita'] = respuestas_llm.citados.apply(len) > 0
respuestas_llm['cita_fuera_contexto'] = [bool(c - ctx) for c, ctx in zip(respuestas_llm.citados, respuestas_llm.contexto)]


def resumir(grupo):
    dentro, fuera = grupo[grupo.en_dominio], grupo[~grupo.en_dominio]
    return pd.Series({
        'citas correctas': dentro.cita_correcta.mean(),
        'respuestas con cita': dentro.con_cita.mean(),
        'citas fuera del contexto': grupo.loc[grupo.con_cita, 'cita_fuera_contexto'].mean(),
        'abstención fuera de dominio': fuera.abstencion.mean(),
        'abstención indebida': dentro.abstencion.mean(),
        'latencia media (s)': grupo.latencia_s.mean(),
    })


comparacion_llm = respuestas_llm.groupby('modelo').apply(resumir).loc[LLMS]
comparacion_llm.round(3)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={'width_ratios': [4, 1]})
comparacion_llm.drop(columns='latencia media (s)').T.plot.bar(ax=ejes[0], rot=20)
ejes[0].set_ylim(0, 1)
ejes[0].set_title('Calidad de las respuestas por modelo')
ejes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
comparacion_llm['latencia media (s)'].plot.bar(ax=ejes[1], rot=0, color=['#4C72B0', '#DD8452'])
ejes[1].set_title('Latencia media (s)')
plt.tight_layout()
plt.show()

Revisamos también algunas respuestas lado a lado: una sobre multas, una sobre comportamiento y una que el código no responde.

In [ ]:
for qid in [35, 16, 49]:
    print('=' * 100)
    print(preguntas.loc[preguntas.id == qid, 'pregunta'].iloc[0], '| correctos:', sorted(gold_por_id[qid]) or '—')
    for modelo in LLMS:
        fila = respuestas_llm[(respuestas_llm.modelo == modelo) & (respuestas_llm.id == qid)].iloc[0]
        print(f'\n[{modelo}] citó {sorted(fila.citados) or "nada"} | contexto {sorted(fila.contexto)}')
        print(fila.respuesta)

In [ ]:
LLM_FINAL = (comparacion_llm.assign(latencia=-comparacion_llm['latencia media (s)'])
             .sort_values(['citas correctas', 'abstención fuera de dominio', 'latencia'], ascending=False).index[0])
print('Modelo generador elegido para el chatbot:', LLM_FINAL)

---
## 12. Umbral de abstención

La instrucción de "no inventar" depende de que el modelo la obedezca. Queremos una segunda barrera que no dependa del modelo: si ni siquiera el fragmento más relevante tiene un puntaje alto, lo más probable es que la pregunta no esté en el código, y es mejor no llamar al generador.

Para calibrarlo tomamos el puntaje máximo de cada pregunta con la configuración final y comparamos su distribución entre las preguntas del código y las de fuera. Como umbral elegimos el valor que maximiza la **exactitud balanceada**: el promedio entre la fracción de preguntas del código que se aceptan y la fracción de preguntas externas que se rechazan. Si hay empate, preferimos el umbral más bajo, para no rechazar preguntas válidas.

Luego combinamos el umbral con las respuestas del modelo elegido: una respuesta se cuenta como abstención si el umbral la bloquea o si el modelo mismo dice que no encontró la información. Como los modelos trabajan con temperatura 0, las respuestas que pasan el umbral son las mismas que ya obtuvimos, y no hace falta volver a generarlas.

In [ ]:
columna_puntaje = 'puntaje_reranker' if RERANKER_FINAL else 'similitud'
maximos = (candidatos[(candidatos.embedding == EMB_FINAL) & (candidatos.estrategia == ESTRATEGIA_FINAL)]
           .groupby('id')[columna_puntaje].max().rename('puntaje_max').reset_index()
           .merge(preguntas[['id', 'en_dominio']], on='id'))

candidatos_umbral = np.unique(maximos.puntaje_max)
evaluacion_umbral = pd.DataFrame({
    'umbral': candidatos_umbral,
    'aceptadas_en_dominio': [(maximos.loc[maximos.en_dominio, 'puntaje_max'] >= u).mean() for u in candidatos_umbral],
    'rechazadas_fuera': [(maximos.loc[~maximos.en_dominio, 'puntaje_max'] < u).mean() for u in candidatos_umbral]})
evaluacion_umbral['exactitud_balanceada'] = evaluacion_umbral[['aceptadas_en_dominio', 'rechazadas_fuera']].mean(axis=1)
UMBRAL = float(evaluacion_umbral.sort_values(['exactitud_balanceada', 'umbral'], ascending=[False, True]).umbral.iloc[0])
print(f'Umbral elegido sobre {columna_puntaje}: {UMBRAL:.4f}')
print(evaluacion_umbral.loc[evaluacion_umbral.umbral == UMBRAL].round(3).to_string(index=False))

fig, eje = plt.subplots(figsize=(9, 3.8))
bins = np.linspace(maximos.puntaje_max.min(), maximos.puntaje_max.max(), 30)
eje.hist(maximos.loc[maximos.en_dominio, 'puntaje_max'], bins=bins, alpha=0.7, label='Preguntas del código')
eje.hist(maximos.loc[~maximos.en_dominio, 'puntaje_max'], bins=bins, alpha=0.7, label='Preguntas por fuera')
eje.axvline(UMBRAL, color='black', linestyle='--', label=f'Umbral = {UMBRAL:.3f}')
eje.set_xlabel(f'Puntaje máximo ({columna_puntaje})')
eje.set_ylabel('Preguntas')
eje.set_title('Puntaje del fragmento más relevante')
eje.legend()
plt.tight_layout()
plt.show()

In [ ]:
final = respuestas_llm[respuestas_llm.modelo == LLM_FINAL].merge(maximos[['id', 'puntaje_max']], on='id',
                                                                  suffixes=('_ctx', ''))
final['bloqueada_por_umbral'] = final.puntaje_max < UMBRAL
final['abstencion_final'] = final.bloqueada_por_umbral | final.abstencion
final['cita_correcta_final'] = final.cita_correcta & ~final.bloqueada_por_umbral

efecto_umbral = pd.DataFrame({
    'sin umbral': [final.loc[final.en_dominio, 'cita_correcta'].mean(),
                   final.loc[~final.en_dominio, 'abstencion'].mean(),
                   final.loc[final.en_dominio, 'abstencion'].mean()],
    'con umbral': [final.loc[final.en_dominio, 'cita_correcta_final'].mean(),
                   final.loc[~final.en_dominio, 'abstencion_final'].mean(),
                   final.loc[final.en_dominio, 'abstencion_final'].mean()]},
    index=['citas correctas', 'abstención fuera de dominio', 'abstención indebida']).round(3)
print(f'Modelo: {LLM_FINAL}')
display(efecto_umbral)
print('Preguntas del código bloqueadas por el umbral:')
final.loc[final.en_dominio & final.bloqueada_por_umbral, ['id', 'puntaje_max']].merge(
    preguntas[['id', 'pregunta']], on='id')

In [ ]:
recuperador.umbral = UMBRAL
llm_final = crear_llm(LLM_FINAL)
cadena_qa = crear_cadena_qa(llm_final, recuperador)
for pregunta in ['¿Cómo se prepara un sancocho de gallina?', '¿Qué me pasa si me cogen manejando borracho?']:
    print('•', pregunta)
    print(formatear_respuesta(cadena_qa.invoke({'input': pregunta})), '\n')

---
## 13. Conversación con memoria

Hasta aquí cada pregunta es independiente. En una conversación real la gente pregunta cosas como "¿y además me inmovilizan el carro?", que solo se entiende con lo que se dijo antes. Usamos `create_history_aware_retriever`: antes de buscar, el modelo reescribe la última pregunta para que se entienda sin el historial, y con esa pregunta reescrita se hace la recuperación. La respuesta final sí recibe el historial completo.

In [ ]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

PROMPT_REFORMULAR = ChatPromptTemplate.from_messages([
    ('system', 'Dada la conversación y la última pregunta del usuario, reescribe esa pregunta para que se entienda '
               'sin la conversación. No la respondas: devuelve solo la pregunta reescrita.'),
    ('placeholder', '{chat_history}'),
    ('human', '{input}'),
])

PROMPT_CONVERSACION = ChatPromptTemplate.from_messages([
    ('system', 'Eres un asistente que orienta a ciudadanos colombianos sobre el Código Nacional de Tránsito '
               '(Ley 769 de 2002). Responde en español, de forma clara y breve, usando únicamente la información '
               'de los artículos del contexto. Cita siempre los artículos con el formato (Art. N). No inventes '
               'valores, plazos ni artículos. Si el contexto no contiene la respuesta, responde exactamente: '
               f'"{MENSAJE_ABSTENCION}"\n\nContexto:\n{{context}}'),
    ('placeholder', '{chat_history}'),
    ('human', '{input}'),
])

recuperador_con_historial = create_history_aware_retriever(llm_final, recuperador, PROMPT_REFORMULAR)
cadena_conversacional = create_retrieval_chain(
    recuperador_con_historial, con_atajo_de_abstencion(create_stuff_documents_chain(llm_final, PROMPT_CONVERSACION)))
reformulador = PROMPT_REFORMULAR | llm_final | StrOutputParser()

historial = []
for pregunta in ['¿Cuál es la multa por andar sin SOAT?', '¿Y además me pueden inmovilizar el carro?']:
    if historial:
        print('Pregunta reescrita para la búsqueda:', reformulador.invoke({'input': pregunta, 'chat_history': historial}))
    salida = cadena_conversacional.invoke({'input': pregunta, 'chat_history': historial})
    historial += [HumanMessage(content=pregunta), AIMessage(content=salida['answer'])]
    print(f'• {pregunta}\n{formatear_respuesta(salida)}\n')

---
## 14. Chatbot con Gradio

La interfaz tiene dos columnas. A la izquierda está la conversación, y cada respuesta termina con sus referencias. A la derecha, un panel muestra los artículos que se recuperaron para la última pregunta, con su puntaje de relevancia y un extracto del texto, para que quien use el asistente pueda comprobar de dónde sale la respuesta.

Manejamos dos historiales por separado: `lc_history`, con los mensajes de LangChain que necesita la cadena, y el historial de Gradio, que solo sirve para pintar el chat. En Colab, `share=True` genera un enlace público temporal para abrir la interfaz desde el navegador.

In [ ]:
import gradio as gr

lc_history = []


def panel_fuentes(contexto):
    if not contexto:
        return ('### Artículos consultados\n\n_Ningún artículo superó el umbral de relevancia, así que el asistente '
                'no respondió con información del código._')
    lineas = ['### Artículos consultados']
    for i, doc in enumerate(contexto, 1):
        m = doc.metadata
        extracto = doc.page_content.split('\n', 1)[-1][:350].replace('\n', ' ')
        lineas.append(f"**{i}. Art. {m['articulo']}{' – ' + m['nombre'] if m['nombre'] else ''}** · "
                      f"relevancia {m['puntaje']:.3f}\n\n> {extracto}…")
    lineas.append(f'\nFuente: [Ley 769 de 2002 – Función Pública]({FUENTE_URL})')
    return '\n\n'.join(lineas)


def responder(pregunta, chat_history):
    chat_history = chat_history or []
    if not pregunta.strip():
        return '', chat_history, gr.update()
    salida = cadena_conversacional.invoke({'input': pregunta, 'chat_history': lc_history})
    lc_history.extend([HumanMessage(content=pregunta), AIMessage(content=salida['answer'])])
    chat_history = chat_history + [{'role': 'user', 'content': pregunta},
                                   {'role': 'assistant', 'content': formatear_respuesta(salida)}]
    return '', chat_history, panel_fuentes(salida['context'])


def reiniciar():
    lc_history.clear()
    return '', [], ''


with gr.Blocks() as demo:
    gr.Markdown('# Asistente del Código Nacional de Tránsito\n'
                'Responde con base en la Ley 769 de 2002 según la compilación de Función Pública '
                f'(texto descargado el {ley.fecha_descarga.iloc[0]}). Es una herramienta de orientación y '
                '**no reemplaza una asesoría legal**.')
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label='Conversación', height=480)
            msg = gr.Textbox(label='¿Qué quieres saber sobre las normas de tránsito?',
                             placeholder='Escribe tu pregunta aquí y presiona Enter.')
            with gr.Row():
                enviar = gr.Button('Enviar', variant='primary')
                limpiar = gr.Button('Limpiar')
        with gr.Column(scale=2):
            fuentes = gr.Markdown()

    msg.submit(responder, [msg, chatbot], [msg, chatbot, fuentes])
    enviar.click(responder, [msg, chatbot], [msg, chatbot, fuentes])
    limpiar.click(reiniciar, None, [msg, chatbot, fuentes], queue=False)

demo.launch(share=IN_COLAB, inline=True, debug=False)

In [ ]:
demo.close()